# Sesión 4 · Tu primer modelo

**Módulo 2 · Python para IA** · 2 sesiones · primeras líneas de código · herramientas clave

> Hoy juntamos todo lo anterior: cogemos datos reales y **entrenamos un modelo** que aprende solo. Verás que con scikit-learn caben en pocas líneas las ideas de la Sesión 2 (entrenamiento, validación, métricas).

**Al terminar la sesión sabrás:**
- Entrenar un modelo de **clasificación** y otro de **regresión**
- Aplicar la separación entrenamiento / prueba en la práctica
- Evaluar resultados con **métricas** y leer una matriz de confusión
- Visualizar datos y resultados con **matplotlib**

*Resultado de la sesión: entrenas un modelo, lo evalúas y lees sus resultados.*

## 1. scikit-learn: la caja de herramientas del ML

**scikit-learn** (se importa como `sklearn`) es la librería de referencia para Machine Learning clásico. Lo bueno: **todos sus modelos se usan igual**, con el mismo patrón de tres pasos. Apréndete este patrón y sabrás usar decenas de modelos:

1. **Crear** el modelo → `modelo = ModeloQueSea()`
2. **Entrenar** con los datos → `modelo.fit(X_entrenamiento, y_entrenamiento)`
3. **Predecir** sobre datos nuevos → `modelo.predict(X_prueba)`

Recuerda el vocabulario de la Sesión 2: **`X`** son las características (las entradas) e **`y`** es la etiqueta (lo que queremos predecir).

## 2. Clasificación: ¿de qué especie es esta flor?

**Clasificar** = asignar una categoría. Volvemos al dataset *iris*: a partir de las medidas de una flor, el modelo predecirá su **especie** (setosa, versicolor o virginica).

Primero cargamos y preparamos los datos.

In [ ]:
import pandas as pd
from sklearn.datasets import load_iris

iris = load_iris(as_frame=True)
X = iris.data          # las 4 medidas (características)
y = iris.target        # la especie codificada como 0, 1, 2 (etiqueta)

print("Características (X):", list(X.columns))
print("Especies posibles (y):", list(iris.target_names))
X.head()

### Paso clave: separar entrenamiento y prueba

Como vimos en la Sesión 2, **no se evalúa con los datos de estudio**. `train_test_split` parte los datos automáticamente. Reservamos el 20% para el examen final.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,        # 20% para prueba, 80% para entrenar
    random_state=42,      # fija el azar para obtener siempre el mismo reparto
    stratify=y,           # mantiene la proporción de cada especie en ambos grupos
)
print("Ejemplos para entrenar:", len(X_train))
print("Ejemplos para probar  :", len(X_test))

### Entrenar el modelo (los tres pasos)

Usamos un **árbol de decisión**: un modelo que aprende preguntas tipo "¿el pétalo mide más de 2,5 cm?" y es fácil de interpretar.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

modelo = DecisionTreeClassifier(max_depth=3, random_state=42)  # 1. crear
modelo.fit(X_train, y_train)                                   # 2. entrenar
predicciones = modelo.predict(X_test)                          # 3. predecir

print("Predicciones del modelo:", predicciones[:10])
print("Valores reales         :", y_test.values[:10])

## 3. Evaluar con métricas

¿Acierta el modelo? La métrica más sencilla para clasificación es la **exactitud** (*accuracy*): el porcentaje de aciertos.

In [ ]:
from sklearn.metrics import accuracy_score

exactitud = accuracy_score(y_test, predicciones)
print(f"Exactitud: {exactitud:.1%}")    # se muestra como porcentaje

### La matriz de confusión

La exactitud resume en un número, pero esconde *dónde* falla el modelo. La **matriz de confusión** muestra, para cada especie real, qué predijo el modelo. La diagonal son los aciertos; todo lo que está fuera de la diagonal son errores.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_estimator(
    modelo, X_test, y_test, display_labels=iris.target_names, cmap="Blues"
)
plt.title("Matriz de confusión")
plt.show()

### 🟢 Ejercicio 1

Repite el entrenamiento, pero cambia el modelo por un **K-Vecinos** (`KNeighborsClassifier`), otro clasificador clásico. Mide su exactitud y compárala con la del árbol.

```python
from sklearn.neighbors import KNeighborsClassifier
```
*Pista: el patrón crear → fit → predict es idéntico.*

In [ ]:
# TODO: entrena un KNeighborsClassifier y mide su exactitud


**Solución:**

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
pred_knn = knn.predict(X_test)
print(f"Exactitud K-Vecinos: {accuracy_score(y_test, pred_knn):.1%}")

## 4. Regresión: predecir un número

**Regresión** = predecir un valor numérico continuo (un precio, una temperatura, una evolución), no una categoría. Usamos un dataset real sobre evolución de la diabetes: a partir de varios datos clínicos se predice un **indicador numérico de progresión** de la enfermedad.

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split

datos = load_diabetes(as_frame=True)
Xr = datos.data
yr = datos.target      # un número continuo (no una categoría)

Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    Xr, yr, test_size=0.2, random_state=42
)
print("Características disponibles:", list(Xr.columns))
print("Ejemplo de valores a predecir:", yr_train.values[:5])

In [ ]:
from sklearn.linear_model import LinearRegression

regresor = LinearRegression()         # 1. crear
regresor.fit(Xr_train, yr_train)      # 2. entrenar
pred_reg = regresor.predict(Xr_test)  # 3. predecir

# Comparamos predicción vs. realidad en los primeros casos
for real, pred in zip(yr_test.values[:5], pred_reg[:5]):
    print(f"Real: {real:6.1f}   |   Predicho: {pred:6.1f}")

### Métricas de regresión

En regresión no hablamos de "aciertos" sino de **cuánto nos desviamos**. Dos métricas habituales:
- **MAE** (error absoluto medio): de media, ¿cuántas unidades nos equivocamos? Cuanto **menor**, mejor
- **R²** (coeficiente de determinación): de 0 a 1, qué parte de la variación explica el modelo. Cuanto **más cerca de 1**, mejor

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

mae = mean_absolute_error(yr_test, pred_reg)
r2 = r2_score(yr_test, pred_reg)
print(f"MAE (error medio): {mae:.1f} unidades")
print(f"R²: {r2:.3f}")

## 5. Visualizar con matplotlib

Un gráfico vale más que mil métricas. En regresión, un truco muy útil es enfrentar **valores reales vs. predichos**: si el modelo fuese perfecto, todos los puntos caerían sobre la diagonal.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 6))
plt.scatter(yr_test, pred_reg, alpha=0.6, color="#1f77b4")
# Línea diagonal de referencia (predicción perfecta)
lim = [yr_test.min(), yr_test.max()]
plt.plot(lim, lim, "--", color="red", label="predicción perfecta")
plt.xlabel("Valor real")
plt.ylabel("Valor predicho")
plt.title("Regresión: real vs. predicho")
plt.legend()
plt.show()

### 🟢 Ejercicio 2

Vuelve a la **clasificación** de iris. Dibuja un gráfico de dispersión (`plt.scatter`) con el largo de pétalo en el eje X y el ancho de pétalo en el eje Y, **coloreando cada punto según su especie**. ¿Se ven grupos separados?

*Pista: recorre las especies con un bucle `for` y dibuja un `scatter` por especie, como hiciste con datos en la Sesión 3.*

In [ ]:
# TODO: gráfico de dispersión de iris coloreado por especie
# (recupera el DataFrame df con la columna 'especie' de la Sesión 3 si lo necesitas)


**Solución:**

In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris

iris = load_iris(as_frame=True)
df = iris.frame.copy()
df["especie"] = df["target"].map(dict(enumerate(iris.target_names)))

plt.figure(figsize=(7, 5))
for especie in df["especie"].unique():
    sub = df[df["especie"] == especie]
    plt.scatter(sub["petal length (cm)"], sub["petal width (cm)"], label=especie, alpha=0.7)
plt.xlabel("Largo de pétalo (cm)")
plt.ylabel("Ancho de pétalo (cm)")
plt.title("Iris: las especies forman grupos separados")
plt.legend()
plt.show()

## 6. Resumen de la sesión y cierre del Módulo 2

- Todo modelo de scikit-learn sigue el patrón **crear → `fit` → `predict`**
- **Clasificación** predice categorías (especie); **regresión** predice números (un indicador)
- Evaluamos sobre datos de **prueba**: con *exactitud* y *matriz de confusión* en clasificación; con *MAE* y *R²* en regresión
- **matplotlib** te ayuda a ver los datos y a juzgar el modelo de un vistazo

Ya sabes escribir Python, manejar datos y entrenar modelos clásicos. **Eso es una base sólida de IA aplicada.**

**Para la próxima sesión (Módulo 3):** damos el salto a los **grandes modelos de lenguaje** (como Claude o GPT): qué son, cómo funcionan por dentro y cómo se les habla bien (*prompt engineering*).

👉 **Piensa una tarea de texto** que te gustaría automatizar (resumir, clasificar, redactar...). La usaremos como ejemplo al trabajar con modelos de lenguaje.